In [ ]:
!pip install -q -U transformers peft datasets accelerate
!pip uninstall -y torchao

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
import torch, json, urllib.request
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER = "Meow07/qwen2.5-0.5b-format-lora"
tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER).eval()

url = "https://raw.githubusercontent.com/Aryan-1947/qwen-format-lora/main/train.json"
train = json.load(urllib.request.urlopen(url))
trained = {r["messages"][0]["content"] for r in train}

raw = load_dataset("HuggingFaceH4/no_robots", split="test")
evalq = [r["prompt"] for r in raw
         if r["category"] in ("Open QA", "Closed QA", "Brainstorm")
         and 20 < len(r["prompt"]) < 300 and r["prompt"] not in trained][:40]
print(len(evalq)); print(evalq[0])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

40
What are 5 things I can do when it's raining in London? I am visiting for the first time and only for a week with my husband. We love to walk, eat good food, and explore.


In [ ]:
SYSTEM = ("You are a helpful assistant. ALWAYS reply in exactly this format and nothing else:\n"
          "Summary: <one sentence>\nKey points:\n- <point>\n- <point>\n- <point>\n"
          "Next step: <one sentence>")

def gen(q, mode):
    msgs = [{"role": "user", "content": q}]
    if mode == "base_prompted":
        msgs = [{"role": "system", "content": SYSTEM}] + msgs
    text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inp = tok(text, return_tensors="pt").to(model.device)
    kw = dict(max_new_tokens=300, do_sample=False, repetition_penalty=1.05,
              pad_token_id=tok.eos_token_id)
    with torch.no_grad():
        if mode == "finetuned":
            out = model.generate(**inp, **kw)
        else:
            with model.disable_adapter():
                out = model.generate(**inp, **kw)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

results = []
for i, q in enumerate(evalq):
    results.append({"q": q,
                    "base": gen(q, "base"),
                    "base_prompted": gen(q, "base_prompted"),
                    "finetuned": gen(q, "finetuned")})
    print(i, end=" ")

json.dump(results, open("eval_v1_answers.json", "w"))
from google.colab import files
files.download("eval_v1_answers.json")

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import re
def ok(a):
    return bool(re.match(r"^Summary:.*\nKey points:\n- .*\n- .*\n- .*\nNext step:", a.strip(), re.S))
for name in ["base", "base_prompted", "finetuned"]:
    print(name, sum(ok(r[name]) for r in results), "/", len(results))

base 0 / 40
base_prompted 0 / 40
finetuned 37 / 40


In [ ]:
import re

# 1) A few prompted answers, to see how close they are
for r in results[:3]:
    print("Q:", r["q"][:80])
    print(r["base_prompted"][:400])
    print("-----")

# 2) A lenient format check
def lenient(a):
    a = a.replace("**", "")
    return ("summary:" in a.lower() and "key points:" in a.lower() and "next step:" in a.lower())

for name in ["base", "base_prompted", "finetuned"]:
    print(name, "lenient:", sum(lenient(r[name]) for r in results), "/", len(results))

# 3) The 3 fine-tuned failures
for r in results:
    if not ok(r["finetuned"]):
        print("FAILED Q:", r["q"][:80])
        print(r["finetuned"][:400])
        print("-----")

Q: What are 5 things I can do when it's raining in London? I am visiting for the fi
1. Bring an umbrella to protect yourself from the rain.
2. Pack waterproof clothing and shoes.
3. Bring a good book or a snack to enjoy while walking.
4. Plan to visit local cafes and restaurants for dining.
5. Take advantage of the weather by exploring the city's parks and gardens.
-----
Q: What other strange and obscure music genres can I check out?
Check out the avant-garde electronic music scene, which blends elements of electronic dance music with avant-garde art forms like performance art and noise art.
-----
Q: Who is Dale Chihuly and what type of art does he produce?
Dale Chihuly is an American glass artist known for his work in the field of glass sculpture, particularly for his large-scale glass sculptures that incorporate elements of nature and other materials.
-----
base lenient: 0 / 40
base_prompted lenient: 0 / 40
finetuned lenient: 40 / 40
FAILED Q: I'm camping in Two Rivers, Minnesota. Wh

In [ ]:
!pip install -q bitsandbytes

In [ ]:
import gc, torch, re, json
del model, base
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
JID = "Qwen/Qwen2.5-7B-Instruct"
jtok = AutoTokenizer.from_pretrained(JID)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
judge = AutoModelForCausalLM.from_pretrained(JID, quantization_config=bnb, device_map="auto")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
RUBRIC = """You are a strict grader. Rate the ANSWER to the QUESTION from 1 to 5.
5 = accurate, relevant and directly useful. 3 = partly useful or partly wrong.
1 = wrong, off-topic or useless.
Judge correctness and relevance only. Do NOT reward length or formatting.
Reply with only the digit."""

def score(q, a):
    msgs = [{"role": "system", "content": RUBRIC},
            {"role": "user", "content": f"QUESTION:\n{q}\n\nANSWER:\n{a}"}]
    text = jtok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inp = jtok(text, return_tensors="pt").to(judge.device)
    with torch.no_grad():
        out = judge.generate(**inp, max_new_tokens=3, do_sample=False)
    s = jtok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"[1-5]", s)
    return int(m.group()) if m else None

for r in results:
    for name in ["base", "base_prompted", "finetuned"]:
        r["score_" + name] = score(r["q"], r[name])

for name in ["base", "base_prompted", "finetuned"]:
    vals = [r["score_" + name] for r in results if r["score_" + name]]
    print(name, "mean:", round(sum(vals) / len(vals), 2), "n =", len(vals))

json.dump(results, open("eval_v1_scored.json", "w"))
from google.colab import files
files.download("eval_v1_scored.json")

base mean: 3.5 n = 40
base_prompted mean: 3.4 n = 40
finetuned mean: 3.48 n = 40


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import random
random.seed(1)
for r in random.sample(results, 6):
    print("Q:", r["q"][:100])
    print("FINETUNED (score", r["score_finetuned"], "):", r["finetuned"][:300])
    print("BASE (score", r["score_base"], "):", r["base"][:300])
    print("-----")

Q: I’m looking for a new city to move to. This city must be in the United States. I don’t want to live 
FINETUNED (score 4 ): Summary: I'm seeking an affordable, suburban location with good amenities and lower crime rates.
Key points:
- Affordable housing options.
- Suburban setting with moderate population density.
- Good quality of life including shopping facilities.
Next step: Research cities that fit these criteria whi
BASE (score 1 ): That sounds like a great choice! Here are some cities that fit your criteria:

1. San Francisco, California - Known for its diverse culture, high quality of life, and affordable living costs.
2. Seattle, Washington - A city with a strong arts scene, vibrant nightlife, and a reputation for being a gr
-----
Q: Which land animal can hold its breath underwater the longest?
FINETUNED (score 3 ): Summary: The blue whale is known for its exceptional ability to hold its breath underwater for extended periods.
Key points:
- Blue whales have been observed hold

In [ ]:
import statistics as st
pairs = [(r["score_finetuned"], r["score_base"]) for r in results
         if r["score_finetuned"] and r["score_base"]]
print("finetuned vs base  win/tie/loss:",
      sum(a > b for a, b in pairs), sum(a == b for a, b in pairs), sum(a < b for a, b in pairs))
d = [a - b for a, b in pairs]
print("mean diff:", round(st.mean(d), 2), "| std error:", round(st.stdev(d) / len(d) ** 0.5, 2))

worst = sorted(results, key=lambda r: r["score_finetuned"] - r["score_base"])[:4]
for r in worst:
    print("\nQ:", r["q"][:150])
    print("FINETUNED (", r["score_finetuned"], "):", r["finetuned"][:350])
    print("BASE      (", r["score_base"], "):", r["base"][:350])

finetuned vs base  win/tie/loss: 12 14 14
mean diff: -0.03 | std error: 0.21

Q: Who is Dale Chihuly and what type of art does he produce?
FINETUNED ( 3 ): Summary: Dale Chihuly is an American glass artist known for his unique techniques using recycled materials to create sculptures.
Key points:
- Dale Chihuly is an American glass artist.
- He specializes in creating large-scale glass sculptures.
- His work often incorporates recycled glass bottles into his designs.
Next step: To learn more about his 
BASE      ( 5 ): Dale Chihuly is an American artist known for his work in glass sculpture. He is best known for his large-scale glass sculptures that have become iconic pieces in the art world. Chihuly's work often incorporates elements of nature, such as plants, flowers, and animals, to create a sense of movement and depth.

Chihuly's primary materials used in his

Q: Who was John Jacob Astor?
FINETUNED ( 3 ): Summary: John Jacob Astor was an American businessman who founded the New York